# Sprints AI-Powered Learner Success & Support Agent
## Notebook 03: NLP, Generative AI & Event-Driven Agent Orchestration (Sprint 4)
**Client:** Sprints (sprints.ai)  
**Track:** AI Pioneers Nationwide Cohort  
**Author:** Mohammed Wael (Lead AI Engineer)

---

### Notebook Overview
This notebook executes **Sprint 4: NLP & Generative AI**:
1. **spaCy Named Entity Recognition (NER)**: Extracting curriculum modules and submission dates from unstructured forum text.
2. **TextBlob Sentiment Assessment**: Measuring polarity and subjectivity to quantify learner frustration.
3. **Deliberative Agent Routing**: Routing distressed queries to human mentors and neutral queries to automated RAG.
4. **Gensim Word2Vec vs FastText Vectorization**: Training dense embeddings and comparing architectural trade-offs.
5. **n8n Workflow Simulation**: Testing event-driven webhook execution and validating the $< 3.5$s SLA.

In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
import spacy
from textblob import TextBlob
from gensim.models import Word2Vec

# Add scripts directory
sys.path.append(os.path.join('..', 'scripts'))
from nlp_pipeline import (
    build_spacy_ner_pipeline,
    extract_entities,
    analyze_sentiment,
    deliberative_agent_router,
    compare_word2vec_fasttext
)

print("NLP and Vectorization modules loaded successfully.")

NLP and Vectorization modules loaded successfully.


### 1. spaCy Custom Named Entity Recognition (NER)
We initialize our enhanced pipeline with `EntityRuler` patterns configured for technical Sprints coursework.

In [2]:
nlp = build_spacy_ner_pipeline()

test_queries = [
    "I am struggling a bit with PyTorch tensors. The module on Oct 12 was too fast for me.",
    "Where is the submission link for the pandas assignment due on Nov 5?",
    "Can someone explain the forget gate in LSTM? The lecture from last Tuesday was cut off.",
    "n8n workflow is failing on the HTTP node, any tips? Ticket submitted on Nov 22nd."
]

for q in test_queries:
    ents = extract_entities(q, nlp_model=nlp)
    print(f"\nQuery: {q}")
    print(" -> Modules Found:", ents["CURRICULUM_MODULE"])
    print(" -> Dates Found:  ", ents["DATE"])


Query: I am struggling a bit with PyTorch tensors. The module on Oct 12 was too fast for me.
 -> Modules Found: ['PyTorch']
 -> Dates Found:   ['Oct 12']

Query: Where is the submission link for the pandas assignment due on Nov 5?
 -> Modules Found: ['pandas']
 -> Dates Found:   ['Nov 5']

Query: Can someone explain the forget gate in LSTM? The lecture from last Tuesday was cut off.
 -> Modules Found: ['LSTM']
 -> Dates Found:   ['last Tuesday']

Query: n8n workflow is failing on the HTTP node, any tips? Ticket submitted on Nov 22nd.
 -> Modules Found: ['n8n']
 -> Dates Found:   ['Nov 22nd']


### 2. TextBlob Sentiment Analysis & Deliberative Agent Logic
Evaluating polarity $[-1.0, 1.0]$ and routing queries accordingly.

In [3]:
queries_to_route = test_queries + [
    "This is the worst experience, none of the code from the notebook works. Thinking of quitting.",
    "Loved the SQL Advanced joins session! Great instructor."
]

for q in queries_to_route:
    res = deliberative_agent_router(q, nlp_model=nlp)
    print(f"\nQuery: {q}")
    print(f" -> Polarity: {res['sentiment']['polarity']} | Subjectivity: {res['sentiment']['subjectivity']}")
    print(f" -> Routing: {res['routing_decision']} (Priority: {res['priority']})")
    print(f" -> Queue:   {res['assigned_queue']}")


Query: I am struggling a bit with PyTorch tensors. The module on Oct 12 was too fast for me.
 -> Polarity: 0.2 | Subjectivity: 0.6
 -> Routing: AUTOMATED_RAG_PIPELINE (Priority: Medium)
 -> Queue:   Sprints_AI_Curriculum_Agent

Query: Where is the submission link for the pandas assignment due on Nov 5?
 -> Polarity: -0.125 | Subjectivity: 0.375
 -> Routing: AUTOMATED_RAG_PIPELINE (Priority: Medium)
 -> Queue:   Sprints_AI_Curriculum_Agent

Query: Can someone explain the forget gate in LSTM? The lecture from last Tuesday was cut off.
 -> Polarity: 0.0 | Subjectivity: 0.0667
 -> Routing: AUTOMATED_RAG_PIPELINE (Priority: Medium)
 -> Queue:   Sprints_AI_Curriculum_Agent

Query: n8n workflow is failing on the HTTP node, any tips? Ticket submitted on Nov 22nd.
 -> Polarity: 0.0 | Subjectivity: 0.0
 -> Routing: AUTOMATED_RAG_PIPELINE (Priority: Medium)
 -> Queue:   Sprints_AI_Curriculum_Agent

Query: This is the worst experience, none of the code from the notebook works. Thinking of quittin

### 3. Gensim Word2Vec Dense Embeddings
Inspecting semantic similarities in the trained Sprints Word2Vec model.

In [4]:
w2v_path = os.path.join('..', 'models', 'word2vec.model')
w2v = Word2Vec.load(w2v_path)
print(f"Loaded Word2Vec model with vocabulary size: {len(w2v.wv)}")

for term in ['pytorch', 'lstm', 'sql', 'pandas']:
    if term in w2v.wv:
        neighbors = w2v.wv.most_similar(term, topn=3)
        print(f"Nearest semantic neighbors to '{term}':", neighbors)

Loaded Word2Vec model with vocabulary size: 363
Nearest semantic neighbors to 'pytorch': [('sql', 0.9275804162025452), ('pandas', 0.9180397391319275), ('rag', 0.9087381958961487)]
Nearest semantic neighbors to 'lstm': [('nltk', 0.892129123210907), ('pandas', 0.8815330266952515), ('pytorch', 0.8755459189414978)]
Nearest semantic neighbors to 'sql': [('pytorch', 0.9275804162025452), ('spacy', 0.912824273109436), ('pandas', 0.897695004940033)]
Nearest semantic neighbors to 'pandas': [('pytorch', 0.9180396199226379), ('spacy', 0.90498286485672), ('rag', 0.9016625285148621)]


### 4. Gensim Word2Vec vs FastText Comparative Benchmark
Analyzing architectural trade-offs: OOV handling, character n-grams, model size, and inference latency.

In [5]:
benchmark = compare_word2vec_fasttext()
df_benchmark = pd.DataFrame(benchmark['dimensions']).T
df_benchmark

,Word2Vec,FastText
Representation Level,Whole-word atomic units (Word-level tokens).,"Subword character n-grams (e.g., 3-to-6-grams)..."
Out-of-Vocabulary (OOV) Handling,Fails completely (KeyError or assigned static ...,Can dynamically synthesize vector embeddings f...
Morphological Richness (MENA / Tech),"Treats 'train', 'training', 'trained', 'traine...",Captures shared prefixes/suffixes and Arabic r...
Memory & Model Size,"Highly compact (e.g., ~1.2 MB for Sprints corp...","Significantly larger (e.g., ~10x-50x larger du..."
Inference & Training Latency,Ultra-fast lookup O(1) via hash table. Sub-mil...,Higher computational overhead per token due to...


### 5. n8n Workflow Simulation & SLA Latency Benchmarking
Simulating the event-driven n8n pipeline executing query parsing, sentiment routing, and knowledge retrieval.

In [6]:
latencies = []
for i in range(10):
    t0 = time.time()
    _ = deliberative_agent_router("Can someone explain the forget gate in LSTM?", nlp_model=nlp)
    latencies.append((time.time() - t0) * 1000)

print(f"Average Pipeline Latency: {np.mean(latencies):.2f} ms")
print(f"Max Latency:              {np.max(latencies):.2f} ms")
print(f"n8n SLA Target (< 3500ms): {'PASSED [OK]' if np.max(latencies) < 3500.0 else 'FAILED'}")

Average Pipeline Latency: 5.12 ms
Max Latency:              6.14 ms
n8n SLA Target (< 3500ms): PASSED [OK]
